# Notebook 01_Data_Extraction_Preprocessing

## Introducción:
El objetivo de este notebook es consolidar una base de datos histórica que combine el ecosistema de activos digitales con indicadores macroeconómicos y de renta variable tradicional.

## Para este estudio, nos centraremos en tres activos:
1. **Bitcoin (BTC):** Activo de referencia en el ecosistema cripto.
2. **NASDAQ (^IXIC):** Índice que representa el sector tecnológico estadounidense, históricamente correlacionado con el apetito por el riesgo.
3. **FED Funds Rate:** Tasa de interés de referencia de la Reserva Federal, utilizada como indicador de la liquidez global y el costo del capital.

## 1. Configuración del Entorno y Modularización del código
Para mantener un código limpio, legible y modular, he separado la lógica económica (`/notebooks`) de la lógica técnica (`/src`). En esta sección, importamos los módulos personalizados que gestionan la conexión con las APIs de Yahoo Finance y FRED.

In [1]:
import sys
import os
from dotenv import load_dotenv

sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_extraction_processing import (
    download_market_data, 
    get_fred_data, 
    calculate_volatility, 
    clean_and_resample
)

print("✅ Funciones importadas correctamente desde src")

✅ Funciones importadas correctamente desde src


## 2. Definición de Parámetros y Seguridad
Utilizamos un archivo `.env` para gestionar la API Key de la FRED (privada) evitando así leakage de datos sensibles. Definimos el horizonte temporal de 10 años.

## Justificación del Periodo (2015 - 2025)
He seleccionado una ventana temporal de 10 años por dos razones críticas:
* **Madurez Institucional:** Antes de 2015, el Bitcoin carecía de la liquidez y la infraestructura necesaria para ser comparado seriamente con los mercados de renta variable, a pesar de que su origen se remonte a inicios de 2009.
* **Ciclos Económicos:** Este periodo cubre multitud de escenarios, desde la fase de tipos de interés cero, pasando por el choque del COVID-19, hasta el ciclo inflacionario (post-Covid) y la subida de tipos de 2023-2024.

In [2]:
load_dotenv() 

api_key = os.getenv('FRED_API_KEY')

if api_key:
    print("✅ API Key cargada exitosamente.")
else:
    print("❌ Error: No se encontró la API Key. Revisa tu archivo .env")

tickers = {'BTC-USD': 'btc', '^IXIC': 'nasdaq'}
start = '2015-01-01'
end = '2025-02-28'

✅ API Key cargada exitosamente.


## 3. Pipeline de extracción y resampling
En este paso, realizamos la descarga de datos vía API y aplicamos las transformaciones oportunas:

1. **Cálculo de Volatilidad:** Estimamos la volatilidad del Bitcoin a partir de sus retornos diarios antes de agrupar los datos.
2. **Resampling Mensual:** Transformamos la frecuencia diaria a Month Start (MS). Esto suaviza el ruido del mercado (propio del ecosistema cripto) y alineamos los datos con la frecuencia de publicación de indicadores macroeconómicos.
*  **IMPORTANTE!** Dado que el NASDAQ cierra los fines de semana pero el BTC no, aplicamos el método de *Forward Fill* (`ffill`) para garantizar la integridad de la serie temporal, rellenando los valores en los que el NASDAQ esta vacio con el último valor disponible.

## Fuentes
Utilizaremos dos fuentes de datos de destacan por su alta fiabilidad:
* **Yahoo Finance (vía `yfinance`):** Para obtener los cierres diarios ajustados de BTC y el NASDAQ.
* **FRED (Federal Reserve Economic Data):** Para obtener la Tasa de Fondos Federales de forma directa de la base de datos de la Reserva Federal a traves de la API oficial.

La integración se realiza mediante módulos personalizados en la carpeta `src` (disponible en el repositorio GitHub), garantizando que el proceso sea reproducible y reciclable.

In [3]:
tickers = {'BTC-USD': 'btc', '^IXIC': 'nasdaq'}
market_raw = download_market_data(tickers, start, end)
fed_raw = get_fred_data('FEDFUNDS', api_key, start, end)

market_raw['btc_vol'] = calculate_volatility(market_raw)
df_final = clean_and_resample(market_raw, fed_raw)

# Guardamos en la carpeta data que creamos
df_final.to_csv('../data/bitcoin_nasdaq_extended_py.csv')
print("✅ Datos guardados en /data")

[*********************100%***********************]  2 of 2 completed


✅ Datos guardados en /data


## Conclusión 
Los datos han sido validados y guardados en la carpeta `/data`. Contamos con una muestra de aproximadamente 122 observaciones mensuales, libre de valores nulos y lista para el análisis descriptivo.

**Siguiente paso:** Proceder al **Notebook_02** para realizar el Análisis Exploratorio de Datos (EDA) e interpretar visualmente información.